In [1]:
import psycopg2
import os
import time
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, Field

load_dotenv()

conn = psycopg2.connect(
    host=os.environ["DB_HOST"],
    port=5432,
    database=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"]
)
cur = conn.cursor()

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

print("Connected to PostgreSQL and Gemini!")

Connected to PostgreSQL and Gemini!


In [2]:
class PillarScore(BaseModel):
    score: int = Field(description="Score from 0 to 10")
    justification: str = Field(description="One sentence explaining specifically why this score was given — reference the scoring guide")
    summary: str = Field(description="2-3 sentences summarising the key ESG points found, covering both strengths and weaknesses")
    evidence_quote: str = Field(description="One short direct quote from the text that best supports the score")

In [4]:
def get_pillar_chunks(filing_id, pillar, max_chunks=10):
    
    pillar_keywords = {
        "E": ["carbon", "emission", "climate", "renewable", "energy", "water", 
              "waste", "sustainability", "net zero", "greenhouse", "fossil", 
              "solar", "wind", "environmental", "pollution", "biodiversity",
              "scope 1", "scope 2", "scope 3", "tcfd", "financed emissions"],
        "S": ["employee", "diversity", "inclusion", "gender", "health", "safety", 
              "community", "human rights", "labour", "labor", "training", 
              "wellbeing", "workforce", "social", "modern slavery", 
              "financial inclusion", "philanthropy", "equity"],
        "G": ["board", "governance", "executive", "compensation", "audit", 
              "compliance", "ethics", "transparency", "shareholder", "director", 
              "risk management", "anti-corruption", "whistleblower",
              "remuneration", "gri", "sasb", "tcfd", "disclosure", "accountability"]
    }
    
    cur.execute("SELECT text FROM chunks WHERE filing_id = %s", (filing_id,))
    all_chunks = [row[0] for row in cur.fetchall()]
    
    keywords = pillar_keywords[pillar]
    relevant = [c for c in all_chunks if any(k in c.lower() for k in keywords)]
    
    return relevant[:max_chunks]

print("Chunk retrieval ready!")

Chunk retrieval ready!


In [5]:
def score_pillar(company_name, filing_id, pillar):
    
    pillar_definitions = {
        "E": """ENVIRONMENTAL (based on GRI 300 series and TCFD framework):
- GHG emissions: Scope 1 (direct), Scope 2 (indirect), Scope 3 (value chain) — targets, reductions, net zero commitments
- Energy: total consumption, renewable energy percentage, energy intensity improvements
- Water: withdrawal, consumption, recycling, water stress areas
- Waste: total waste generated, recycling rate, hazardous waste
- Biodiversity: impact on ecosystems, protected areas
- Climate risk: physical and transition risks, TCFD alignment
- For financial companies specifically: financed emissions, green lending, sustainable finance targets""",

        "S": """SOCIAL (based on GRI 400 series):
- Employment: turnover rate, employee benefits, job creation
- Health & Safety: injury rates, fatality rates, safety programs
- Training & Education: average training hours, skills development programs
- Diversity & Inclusion: gender pay gap, board diversity, minority representation
- Human Rights: supply chain due diligence, modern slavery policies
- Community: local investment, philanthropy, financial inclusion programs
- For financial companies specifically: access to financial services for underserved communities, responsible lending practices""",

        "G": """GOVERNANCE (based on GRI 2 and SASB Financial Sector standards):
- Board structure: size, independence percentage, diversity, separation of CEO and Chair
- Executive compensation: CEO pay ratio, ESG-linked remuneration, bonus structure
- Anti-corruption: policies, training, reported incidents
- Risk management: enterprise risk framework, ESG risk integration
- Audit: external auditor independence, internal audit function
- Transparency: ESG reporting frameworks used (GRI, SASB, TCFD), third-party assurance
- Shareholder rights: voting rights, engagement policy"""
    }
    
    pillar_names = {
        "E": "Environmental",
        "S": "Social",
        "G": "Governance"
    }
    
    chunks = get_pillar_chunks(filing_id, pillar)
    
    if not chunks:
        print(f"  No chunks found for {pillar}")
        return None
    
    combined_text = "\n\n---\n\n".join(chunks)
    
    prompt = f"""You are a senior ESG analyst reviewing {company_name}'s annual report.

Your task: evaluate how well {company_name} performs on the {pillar_names[pillar]} pillar of ESG.

Use ONLY the excerpts provided below as your evidence. Do not use any external knowledge about this company.

--- OFFICIAL DEFINITION OF {pillar_names[pillar].upper()} ---
{pillar_definitions[pillar]}

--- SCORING GUIDE ---
0-2: No meaningful mention or effort found in the report
3-4: Basic mentions but no concrete targets, data or commitments
5-6: Some initiatives with partial data or targets mentioned
7-8: Clear commitments with measurable targets and reported progress
9-10: Industry-leading performance with verified results, ambitious targets and third-party assurance

--- YOUR RESPONSE MUST INCLUDE ---
1. score: integer 0-10 based strictly on the scoring guide above
2. justification: one sentence explaining exactly why this score was given, referencing the scoring guide (e.g. "Score is 7 because the company has concrete net-zero targets with reported progress but lacks Scope 3 data")
3. summary: 2-3 sentences covering what the company does well and what is missing or weak on this pillar
4. evidence_quote: one short direct quote from the excerpts below that best supports your score

--- EXCERPTS FROM {company_name.upper()}'S 10-K FILING ---
{combined_text}

Based strictly on the above excerpts, evaluate {company_name} on the {pillar_names[pillar]} pillar."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": PillarScore,
            "temperature": 0.0
        }
    )
    
    return PillarScore.model_validate_json(response.text)

print("Scoring function ready!")

Scoring function ready!


In [6]:
def save_score(filing_id, pillar, result):
    cur.execute("""
        INSERT INTO esg_scores 
            (filing_id, pillar, score, justification, summary, evidence_quote, model_used)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (
        filing_id,
        pillar,
        result.score,
        result.justification,
        result.summary,
        result.evidence_quote,
        "gemini-2.5-flash"
    ))
    conn.commit()
    print(f"  Saved {pillar} score: {result.score}/10")

print("Save function ready!")

Save function ready!


In [9]:
cur.execute("""
    SELECT f.id FROM filings f
    JOIN companies c ON f.company_id = c.id
    WHERE c.name = 'JPMorgan'
""")
filing_id = cur.fetchone()[0]
print(f"JPMorgan filing_id: {filing_id}\n")

for pillar in ["E", "S", "G"]:
    print(f"\n--- {pillar} ---")
    result = score_pillar("JPMorgan", filing_id, pillar)
    if result:
        print(f"Score:         {result.score}/10")
        print(f"Justification: {result.justification}")
        print(f"Summary:       {result.summary}")
        print(f"Evidence:      {result.evidence_quote}")
        save_score(filing_id, pillar, result)
    time.sleep(2)

print("\nVerify in database:")
cur.execute("""
    SELECT pillar, score, justification, summary
    FROM esg_scores 
    WHERE filing_id = %s
""", (filing_id,))
for row in cur.fetchall():
    print(f"\n{row[0]}: {row[1]}/10")
    print(f"  Justification: {row[2]}")
    print(f"  Summary: {row[3][:80]}...")

JPMorgan filing_id: 15


--- E ---
Score:         4/10
Justification: Score is 4 because the report acknowledges physical and transition climate risks and mentions alternative energy investments, but provides no concrete targets, data, or commitments for environmental performance or sustainable finance.
Summary:       JPMorgan acknowledges climate change as a strategic risk, identifying both physical and transition risks that could impact its business and clients. The company also mentions involvement in alternative energy tax-oriented investments. However, the report lacks specific environmental performance data, measurable targets for emissions, energy, water, or waste, and detailed commitments or strategies for sustainable finance beyond a single mention.
Evidence:      Both physical risks and transition risks associated with climate change could negatively impact JPMorganChase and its clients and customers.
  Saved E score: 4/10

--- S ---
Score:         5/10
Justification: Score i

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [10]:
cur.execute("""
    SELECT c.name, f.id 
    FROM companies c
    JOIN filings f ON f.company_id = c.id
    ORDER BY c.name
""")
all_companies = cur.fetchall()

print(f"Scoring {len(all_companies)} companies...\n")

SLEEP_BETWEEN_CALLS = 5
SLEEP_BETWEEN_COMPANIES = 8

success = 0
skipped = 0
failed = 0

for company_name, filing_id in all_companies:
    
    cur.execute("""
        SELECT pillar FROM esg_scores WHERE filing_id = %s
    """, (filing_id,))
    already_scored = [row[0] for row in cur.fetchall()]
    
    pillars_to_score = [p for p in ["E", "S", "G"] if p not in already_scored]
    
    if not pillars_to_score:
        print(f"✓ Skipping {company_name} — all pillars already scored")
        skipped += 1
        continue
    
    print(f"\n{company_name} — scoring {pillars_to_score}:")
    
    for pillar in pillars_to_score:
        max_retries = 5
        for attempt in range(max_retries):
            try:
                result = score_pillar(company_name, filing_id, pillar)
                if result:
                    save_score(filing_id, pillar, result)
                    success += 1
                    time.sleep(SLEEP_BETWEEN_CALLS)
                    break
                    
            except Exception as e:
                print(f"  ⚠ Attempt {attempt+1}/{max_retries} failed: {e}")
                print(f"  Retrying in 15 seconds...")
                time.sleep(15)
        else:
            print(f"  ✗ Failed {pillar} after {max_retries} attempts, moving on")
            failed += 1
    
    conn.commit()
    print(f"  ✓ {company_name} saved")
    time.sleep(SLEEP_BETWEEN_COMPANIES)

print(f"\n--- DONE ---")
print(f"Scored:  {success}")
print(f"Skipped: {skipped}")
print(f"Failed:  {failed}")

Scoring 28 companies...

✓ Skipping Aflac — all pillars already scored
✓ Skipping Allstate — all pillars already scored
✓ Skipping AmericanExpress — all pillars already scored
✓ Skipping BankOfAmerica — all pillars already scored
✓ Skipping BerkshireHathaway — all pillars already scored
✓ Skipping BlackRock — all pillars already scored
✓ Skipping CapitalOne — all pillars already scored
✓ Skipping CharlesSchwab — all pillars already scored
✓ Skipping Citigroup — all pillars already scored
✓ Skipping CMEGroup — all pillars already scored
✓ Skipping DiscoverFinancial — all pillars already scored
✓ Skipping Fiserv — all pillars already scored
✓ Skipping Goldman — all pillars already scored
✓ Skipping ICE — all pillars already scored
✓ Skipping JPMorgan — all pillars already scored
✓ Skipping Mastercard — all pillars already scored
✓ Skipping MetLife — all pillars already scored
✓ Skipping MorganStanley — all pillars already scored
✓ Skipping Nasdaq — all pillars already scored
✓ Skipping P

In [11]:
cur.execute("""
    SELECT c.name, 
           MAX(CASE WHEN e.pillar = 'E' THEN e.score END) as E,
           MAX(CASE WHEN e.pillar = 'S' THEN e.score END) as S,
           MAX(CASE WHEN e.pillar = 'G' THEN e.score END) as G
    FROM companies c
    JOIN filings f ON f.company_id = c.id
    JOIN esg_scores e ON e.filing_id = f.id
    GROUP BY c.name
    ORDER BY c.name;
""")

rows = cur.fetchall()
print(f"{'Company':<25} {'E':>5} {'S':>5} {'G':>5}")
print("-" * 40)
for row in rows:
    print(f"{row[0]:<25} {str(row[1]):>5} {str(row[2]):>5} {str(row[3]):>5}")

print(f"\nTotal companies scored: {len(rows)}")

Company                       E     S     G
----------------------------------------
Aflac                         6     5     7
Allstate                      3     3     7
AmericanExpress               4     6     7
BankOfAmerica                 4     7     6
BerkshireHathaway             6     4     5
BlackRock                     3     3     7
CapitalOne                    3     5     6
CharlesSchwab                 3     3     6
Citigroup                     4     1     6
CMEGroup                      5     3     7
DiscoverFinancial             3     6     5
Fiserv                        0     6     7
Goldman                       7     4     6
ICE                           0     5     7
JPMorgan                      4     5     6
Mastercard                    0     0     2
MetLife                       6     3     6
MorganStanley                 4     4     7
Nasdaq                        5     3     6
PayPal                        4     6     6
PNCFinancial                  4    

In [12]:
cur.close()
conn.close()
print("Connection closed!")

Connection closed!
